In [128]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression

In [129]:
model = LogisticRegression()

In [130]:
df = pd.read_csv('spam.csv', encoding='latin1')

In [131]:
df.head()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


In [132]:
df.columns

Index(['v1', 'v2', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4'], dtype='object')

In [133]:
print(df.columns)

Index(['v1', 'v2', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4'], dtype='object')


In [134]:
df.head()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


In [135]:
df.isnull().sum()

,0
v1,0
v2,0
Unnamed: 2,5522
Unnamed: 3,5560
Unnamed: 4,5566


In [136]:
df = df[['v1', 'v2']]

In [137]:
# rename properly
df.columns = ['label', 'text']

In [138]:
df.head()
df.isnull().sum()

,0
label,0
text,0


In [139]:
df['label'] = df['label'].map({'ham': 0, 'spam': 1})

In [140]:
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r'\d+', '', text)        # ❗ remove numbers
    text = re.sub(r'\W', ' ', text)        # remove special chars
    text = re.sub(r'\s+', ' ', text)       # remove extra spaces
    return text

df['text'] = df['text'].apply(clean_text)

In [141]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    stop_words='english',
    min_df=2,              # remove rare words
    token_pattern=r'\b[a-zA-Z]{2,}\b'   # ❗ only real words (no numbers)
)
X = vectorizer.fit_transform(df['text'])
y = df['label']

In [142]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,      # 80% train, 20% test
    random_state=42     # for reproducibility
)

In [143]:
print(X_train.shape)
print(X_test.shape)

(4457, 3559)
(1115, 3559)


In [144]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

LogisticRegression(max_iter=1000)

In [145]:
y_pred = model.predict(X_test)

In [146]:
from sklearn.metrics import accuracy_score, classification_report

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.9542600896860987
              precision    recall  f1-score   support

           0       0.95      1.00      0.97       965
           1       0.97      0.68      0.80       150

    accuracy                           0.95      1115
   macro avg       0.96      0.84      0.89      1115
weighted avg       0.96      0.95      0.95      1115



In [157]:
def predict_spam(text):
    text_vector = vectorizer.transform([text])
    prediction = model.predict(text_vector)
    return "Spam" if prediction[0] == 1 else "Not Spam"

print(predict_spam("🎉 Congratulations! You’ve been selected as a winner of a FREE iPhone 15! Click the link below to claim your prize now before it expires: 👉 http://claim-now-free.com Hurry! Limited time offer ⏳"))

Spam


In [148]:
feature_names = vectorizer.get_feature_names_out()
coefficients = model.coef_[0]

In [149]:
import pandas as pd

weights_df = pd.DataFrame({
    'word': feature_names,
    'weight': coefficients
})

weights_df.sort_values(by='weight', ascending=False).head(10)

,word,weight
3226,txt,4.737312
1943,mobile,3.861977
529,claim,3.773928
1121,free,3.697825
3506,www,3.413123
2920,stop,3.365046
3239,uk,3.342321
2522,reply,2.966768
2062,new,2.907685
2690,service,2.886701
